In [53]:
from pathlib import Path
from scipy.io import loadmat
import pandas as pd
import numpy as np

In [54]:
data_folder = Path("DD_data")
mat_files = sorted(data_folder.glob("*.mat"))

In [55]:
# get colummns of the dataframes
data_0 = loadmat(mat_files[0])
data_labels = data_0["data_labels"]
columns = [label[0] for label in data_labels[0]]

print(columns)

['immOutcome', 'delOutcome', 'delay', 'action (1=immediate, 2=delayed, 0=missing)', 'p_imm', 'condition (1=reward,2=loss)', 'RT']


In [56]:
# convert data to dataframes
participants = []

for file in mat_files:
    data = loadmat(file)

    train_df = pd.DataFrame(data["data_train"], columns=columns)
    test_df = pd.DataFrame(data["data_test"], columns=columns)

    participants.append({
        "train": train_df,
        "test": test_df
    })

In [57]:
participants[0]["train"]

,immOutcome,delOutcome,delay,"action (1=immediate, 2=delayed, 0=missing)",p_imm,"condition (1=reward,2=loss)",RT
0,1.79,5.0,180.0,1.0,0.5,1.0,1721.1
1,9.99,10.0,30.0,1.0,0.5,1.0,2190.2
2,7.33,10.0,365.0,1.0,0.5,1.0,1590.0
3,0.46,50.0,180.0,2.0,0.5,1.0,1804.1
4,0.91,50.0,90.0,2.0,0.5,1.0,1722.5
...,...,...,...,...,...,...,...
75,48.54,50.0,30.0,1.0,0.5,1.0,3756.2
76,4.59,5.0,90.0,1.0,0.5,1.0,1209.4
77,10.53,20.0,90.0,1.0,0.5,1.0,2396.9
78,9.99,10.0,90.0,1.0,0.5,1.0,1274.4


In [58]:
participants[0]["test"]

,immOutcome,delOutcome,delay,"action (1=immediate, 2=delayed, 0=missing)",p_imm,"condition (1=reward,2=loss)",RT
0,4.91,20.0,365.0,1.0,0.5,1.0,1507.5
1,23.27,50.0,90.0,1.0,0.7,1.0,1260.5
2,2.55,20.0,90.0,2.0,0.2,1.0,2113.5
3,15.90,50.0,180.0,1.0,0.5,1.0,1764.1
4,16.18,20.0,30.0,2.0,0.8,1.0,1984.3
...,...,...,...,...,...,...,...
175,16.64,50.0,90.0,1.0,0.3,1.0,1741.6
176,4.99,5.0,365.0,2.0,0.8,1.0,1405.7
177,7.88,10.0,365.0,1.0,0.8,1.0,1249.0
178,3.58,5.0,90.0,1.0,0.6,1.0,2584.1


In [59]:
# check missing and invalid data for all participants

action_zero = 0
imm_nan = 0
del_nan = 0
delay_nan = 0
action_nan = 0
condition_nan = 0
rt_nan = 0
rt_nonpositive = 0

for participant in participants:
    for run in ["train", "test"]:
        df = participant[run]

        action_zero += (df["action (1=immediate, 2=delayed, 0=missing)"] == 0).sum()

        imm_nan += df["immOutcome"].isna().sum()
        del_nan += df["delOutcome"].isna().sum()
        delay_nan += df["delay"].isna().sum()
        action_nan += df["action (1=immediate, 2=delayed, 0=missing)"].isna().sum()
        condition_nan += df["condition (1=reward,2=loss)"].isna().sum()
        rt_nan += df["RT"].isna().sum()

        rt_nonpositive += (df["RT"] <= 0).sum()

print("action = 0:", action_zero)
print("immOutcome NaN:", imm_nan)
print("delOutcome NaN:", del_nan)
print("delay NaN:", delay_nan)
print("action NaN:", action_nan)
print("condition NaN:", condition_nan)
print("RT NaN:", rt_nan)
print("RT <= 0:", rt_nonpositive)


action = 0: 93
immOutcome NaN: 60
delOutcome NaN: 0
delay NaN: 0
action NaN: 0
condition NaN: 0
RT NaN: 93
RT <= 0: 0


In [60]:
# This is the cleaned data that will be used for analysis.

cleaned_data = []

In [61]:
# record how many rows are deleted from each run
deleted_train = 0
deleted_test = 0

for i, participant in enumerate(participants):

    train_df = participant["train"].copy()
    test_df = participant["test"].copy()

    original_train = len(train_df)
    original_test = len(test_df)

    # remove missing choices
    train_df = train_df[
        train_df["action (1=immediate, 2=delayed, 0=missing)"] != 0
    ]

    test_df = test_df[
        test_df["action (1=immediate, 2=delayed, 0=missing)"] != 0
    ]

    # remove NaNs
    nan_columns = [
        "immOutcome",
        "delOutcome",
        "delay",
        "action (1=immediate, 2=delayed, 0=missing)",
        "condition (1=reward,2=loss)",
        "RT"
    ]

    train_df = train_df.dropna(subset=nan_columns).copy()
    test_df = test_df.dropna(subset=nan_columns).copy()

    # create binary choice
    train_df["action(cleaned)"] = (
        train_df["action (1=immediate, 2=delayed, 0=missing)"] == 1
    ).astype(int)

    test_df["action(cleaned)"] = (
        test_df["action (1=immediate, 2=delayed, 0=missing)"] == 1
    ).astype(int)

    deleted_train += original_train - len(train_df)
    deleted_test += original_test - len(test_df)

    cleaned_data.append({
        "participant_id": i + 1,
        "train": train_df,
        "test": test_df
    })

print("deleted from Train:", deleted_train)
print("deleted from Test:", deleted_test)

deleted from Train: 34
deleted from Test: 118


In [62]:
# check missing and invalid data for all participants

action_zero = 0
imm_nan = 0
del_nan = 0
delay_nan = 0
action_nan = 0
condition_nan = 0
rt_nan = 0
rt_nonpositive = 0

for participant in cleaned_data:
    for run in ["train", "test"]:
        df = participant[run]

        action_zero += (df["action (1=immediate, 2=delayed, 0=missing)"] == 0).sum()

        imm_nan += df["immOutcome"].isna().sum()
        del_nan += df["delOutcome"].isna().sum()
        delay_nan += df["delay"].isna().sum()
        action_nan += df["action (1=immediate, 2=delayed, 0=missing)"].isna().sum()
        condition_nan += df["condition (1=reward,2=loss)"].isna().sum()
        rt_nan += df["RT"].isna().sum()

print("action = 0:", action_zero)
print("immOutcome NaN:", imm_nan)
print("delOutcome NaN:", del_nan)
print("delay NaN:", delay_nan)
print("action NaN:", action_nan)
print("condition NaN:", condition_nan)
print("RT NaN:", rt_nan)


action = 0: 0
immOutcome NaN: 0
delOutcome NaN: 0
delay NaN: 0
action NaN: 0
condition NaN: 0
RT NaN: 0


In [63]:
cleaned_data[0]["train"].head()

,immOutcome,delOutcome,delay,"action (1=immediate, 2=delayed, 0=missing)",p_imm,"condition (1=reward,2=loss)",RT,action(cleaned)
0,1.79,5.0,180.0,1.0,0.5,1.0,1721.1,1
1,9.99,10.0,30.0,1.0,0.5,1.0,2190.2,1
2,7.33,10.0,365.0,1.0,0.5,1.0,1590.0,1
3,0.46,50.0,180.0,2.0,0.5,1.0,1804.1,0
4,0.91,50.0,90.0,2.0,0.5,1.0,1722.5,0
